# D9 — Semantic, Narrative & Pragmatic Complexity

This notebook extracts **25 numeric features** per document

### Feature inventory (25 numeric)

| ID  | Sub-dimension                       | Feature                              | Source / method                                                  |
|-----|-------------------------------------|--------------------------------------|------------------------------------------------------------------|
| 1   | D9.1 Lexical–Semantic Density       | information_density_content_words    | Halliday (1985) lexical density                                  |
| 2   |                                     | propositional_density                | CPIDR algorithm (Brown et al. 2008) via `ideadensity`            |
| 3   |                                     | mean_concreteness                    | Brysbaert et al. (2014) 40k-lemma norms                          |
| 4   |                                     | mean_imageability                    | MRC Psycholinguistic Database (Coltheart 1981)                   |
| 5   | D9.2 Figurative Language            | simile_density                       | spaCy `Matcher` over standard comparison patterns (Niculae & Danescu-Niculescu-Mizil 2014) |
| 6   |                                     | metaphor_density                     | XLM-R fine-tuned on VUA Metaphor Corpus (Steen et al. 2010; Wachowiak & Gromann 2023) |
| 7   |                                     | personification_density              | PINEAPPLE verb-personification scores (Keh 2022)                 |
| 8   | D9.3 Narrative Structure            | character_count                      | spaCy NER `PERSON` ∪ `PROPN` head-of-noun-chunk deduplication    |
| 9   |                                     | character_introduction_rate          | New `PERSON` entities per sentence                               |
| 10  |                                     | story_arc_complexity                 | Std of VADER per-sentence compound sentiment (Reagan et al. 2016) |
| 11  | D9.4 Temporal & Tense Structure     | chronological_order_score            | Tense-sequence + backward-marker heuristic                       |
| 12  |                                     | temporal_expression_density          | spaCy DATE/TIME entities ∪ temporal adverb lexicon (per sentence) |
| 13  |                                     | tense_consistency                    | Proportion of finite verbs sharing the dominant tense            |
| 14  |                                     | tense_switch_frequency               | Tense switches / (n_finite_verbs − 1)                            |
| 15  | D9.5 Register & Formality           | formality_score                      | Heylighen & Dewaele (1999) F-score from POS frequencies          |
| 16  |                                     | contraction_density                  | Regex over standard English contraction inventory                |
| 17  |                                     | first_person_pronoun_density         | Biber (1988) involved-vs-informational dimension                 |
| 18  |                                     | second_person_pronoun_density        | Biber (1988)                                                     |
| 19  |                                     | third_person_pronoun_density         | Biber (1988)                                                     |
| 20  | D9.6 Pragmatic Inference Load       | question_density                     | Sentence-final `?` proportion                                    |
| 21  |                                     | exclamation_density                  | Sentence-final `!` proportion                                    |
| 22  |                                     | imperative_density                   | Root VERB with no nominal subject (Quirk et al. 1985)            |
| 23  |                                     | implicature_marker_density           | Hyland (2005) metadiscourse + Mann & Thompson (1988) RST markers |
| 24  | D9.7 Cultural & Domain Knowledge    | cultural_entity_density              | Bhasha-Wiki-Indic (~250k Indian-context entities, freq ≥ 3)      |
| 25  |                                     | domain_vocab_ratio                   | Zipf frequency < 3.0 via `wordfreq` (Van Heuven et al. 2014)     |

### Pipeline architecture

Two memory-isolated passes — **Pass 1** runs spaCy + 24 features over the corpus,
**Pass 2** loads the metaphor model only after spaCy is freed and computes the
25th feature (`metaphor_density`).

All external resources (MRC, Brysbaert, PINEAPPLE, Bhasha-Wiki-Indic) are built
once and pickled to `./d9_cache/`; subsequent runs reload in milliseconds.

## 1 · Install dependencies

In [ ]:
# Core NLP
!pip install -q spacy vaderSentiment ideadensity wordfreq
!python -m spacy download en_core_web_sm -q

# Psycholinguistic norms + Bhasha-Wiki
!pip install -q datasets openpyxl xlrd

# Metaphor model
!pip install -q transformers torch

print('All installs done.')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 32.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All installs done.


## 2 · Imports & NLP backbone

In [ ]:
import re, os, sys, pickle, requests, warnings, gc
from pathlib import Path
from collections import Counter, defaultdict
from time import time

import numpy as np
import pandas as pd

import spacy
from spacy.matcher import Matcher
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

# Single shared spaCy pipeline — parser ON (needed for personification,
# imperative detection, dep-based imperative cue).
nlp   = spacy.load('en_core_web_sm')
vader = SentimentIntensityAnalyzer()
print(f'spaCy pipeline: {nlp.pipe_names}')

# Cache directory — keep all reusable artefacts here
CACHE_DIR = Path('./d9_cache')
CACHE_DIR.mkdir(exist_ok=True)
print(f'Cache directory: {CACHE_DIR.resolve()}')


spaCy pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
Cache directory: /content/d9_cache


## 3 · Load corpus

Expects an Excel file with at least an `index` column and a text column (default name `content`). Edit `DATA_PATH` and `TEXT_COL` as needed.

In [ ]:
# === EDIT PATH IF NEEDED ===
DATA_PATH = '/content/onestopenglish_dataset.csv'
TEXT_COL  = 'content'
ID_COL    = 'index'

df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = [c.strip() for c in df_raw.columns]

assert ID_COL   in df_raw.columns, f'Missing {ID_COL} column'
assert TEXT_COL in df_raw.columns, f'Missing {TEXT_COL} column'
print(f'Loaded {len(df_raw)} documents from {DATA_PATH}')


Loaded 567 documents from /content/onestopenglish_dataset.csv


## 4 · Cache 1 — Psycholinguistic norms (MRC + Brysbaert 2014)

**Concreteness** — Brysbaert et al. (2014) report mean concreteness ratings on a
1–5 scale for ~40,000 English lemmas. We supplement with MRC concreteness
(Coltheart 1981; native scale 100–700, rescaled to 1–5) for words not in
Brysbaert. Where both norms exist, Brysbaert takes precedence (larger, more
recent, more reliable inter-rater agreement).

**Imageability** — MRC `Imageability` field, native scale 100–700, used as-is.

In [ ]:
PSYCH_CACHE = CACHE_DIR / 'psycholing_cache.pkl'

def build_psycholing_cache():
    print('Building psycholinguistic cache (one-time)...')
    from datasets import load_dataset

    print('  Loading MRC (Coltheart 1981)...')
    ds = load_dataset('StephanAkkerman/MRC-psycholinguistic-database', split='train')
    df_mrc = ds.to_pandas()
    mrc_conc_raw = dict(zip(df_mrc['Word'].str.lower(), df_mrc['Concreteness']))
    mrc_img_dict = dict(zip(df_mrc['Word'].str.lower(), df_mrc['Imageability']))

    print('  Loading Brysbaert et al. 2014 from OSF...')
    try:
        df_b = pd.read_excel('https://osf.io/download/3r2v8/')
        df_b.columns = [c.strip() for c in df_b.columns]
        w_col = [c for c in df_b.columns if 'word' in c.lower()][0]
        c_col = [c for c in df_b.columns if 'conc' in c.lower()][0]
        brysbaert_dict = dict(zip(df_b[w_col].str.lower(), df_b[c_col]))
    except Exception as e:
        print(f'    Brysbaert OSF download failed ({e}); MRC only.')
        brysbaert_dict = {}

    # Rescale MRC 100–700 → 1–5, then let Brysbaert override (larger + more recent).
    merged_conc = {w: 1 + (v - 100) / 600 * 4 for w, v in mrc_conc_raw.items()}
    merged_conc.update(brysbaert_dict)

    cache = {'merged_conc': merged_conc, 'mrc_img_dict': mrc_img_dict}
    with open(PSYCH_CACHE, 'wb') as f:
        pickle.dump(cache, f)
    print(f'  ✓ Cached {len(merged_conc):,} concreteness + {len(mrc_img_dict):,} imageability entries')
    return cache

if PSYCH_CACHE.exists():
    with open(PSYCH_CACHE, 'rb') as f:
        _psych = pickle.load(f)
    print(f'✓ Loaded psycholing cache ({len(_psych["merged_conc"]):,} concreteness entries)')
else:
    _psych = build_psycholing_cache()

merged_conc  = _psych['merged_conc']
mrc_img_dict = _psych['mrc_img_dict']


✓ Loaded psycholing cache (119,380 concreteness entries)


## 5 · Cache 2 — PINEAPPLE personification verb scores

PINEAPPLE (Keh 2022) is a corpus of paired sentences, where each pair has a
literal sentence and its personified counterpart with the *same* verb. From the
sentences in `depersonification_pers.txt` (personifying uses) and
`depersonification_literal.txt` (literal uses), we compute a Laplace-smoothed
empirical estimate
`P(personification | verb v) = (count_pers(v) + 1) / (count_pers(v) + count_literal(v) + 2)`
for every verb that appears in either file. At extraction time, a clause counts
as personification when its head verb scores ≥ 0.65 *and* its grammatical
subject is non-PERSON.

In [ ]:
PINEAPPLE_CACHE = CACHE_DIR / 'pineapple_verb_scores.pkl'
PINEAPPLE_URLS = {
    'pers':    'https://raw.githubusercontent.com/sedrickkeh/PINEAPPLE/master/data/depersonification_pers.txt',
    'literal': 'https://raw.githubusercontent.com/sedrickkeh/PINEAPPLE/master/data/depersonification_literal.txt',
}

def build_pineapple_scores():
    print('Building PINEAPPLE verb scores (one-time, ~1 min)...')
    lines = {}
    for key, url in PINEAPPLE_URLS.items():
        r = requests.get(url, timeout=30); r.raise_for_status()
        lines[key] = r.text.splitlines()
    def extract_verb_counts(sents):
        cnt = Counter()
        for d in nlp.pipe([s.strip() for s in sents if s.strip()], batch_size=64):
            for t in d:
                if t.pos_ == 'VERB':
                    cnt[t.lemma_.lower()] += 1
        return cnt
    p = extract_verb_counts(lines['pers'])
    l = extract_verb_counts(lines['literal'])
    scores = {v: (p[v] + 1) / (p[v] + l[v] + 2) for v in set(p) | set(l)}
    with open(PINEAPPLE_CACHE, 'wb') as f:
        pickle.dump(scores, f)
    print(f'  ✓ Cached {len(scores)} PINEAPPLE verb scores')
    return scores

if PINEAPPLE_CACHE.exists():
    with open(PINEAPPLE_CACHE, 'rb') as f:
        verb_scores = pickle.load(f)
    print(f'✓ Loaded {len(verb_scores)} PINEAPPLE verb scores')
else:
    verb_scores = build_pineapple_scores()

PERS_THRESHOLD = 0.65  # Median split on smoothed P(personification); same threshold as Keh (2022) reference.


✓ Loaded 432 PINEAPPLE verb scores


## 6 · Cache 3 — Indian entity lexicon (Bhasha-Wiki-Indic)

Streams `soketlabs/bhasha-wiki-indic` (English partition) and harvests
capitalised multi-word noun phrases of length ≥ 4 with corpus frequency ≥ 3,
yielding ~250k Indian-context named entities. Streaming keeps peak RAM under
~500 MB (vs ~3–5 GB for the non-streaming materialisation). The frequency cutoff
and pattern are unchanged from the original lexicon construction.

In [ ]:
ENTITY_CACHE = CACHE_DIR / 'indian_entities_lexicon.pkl'

def build_indian_entity_lexicon(stream_cap: int = 260_000, char_cap: int = 1000):
    import gc
    from datasets import load_dataset

    print('=' * 60)
    print('Building Indian Entity Lexicon (one-time, streaming, ~20 min)')
    print('=' * 60)

    ds = load_dataset(
        'soketlabs/bhasha-wiki-indic', '20231101.en',
        split='train', streaming=True,
    )

    entity_pattern = re.compile(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b')
    counter = Counter()
    t0 = time()

    for i, article in enumerate(ds):
        if i >= stream_cap:
            break
        if i % 20_000 == 0 and i:
            elapsed = time() - t0
            print(f'  {i:>7,} articles   elapsed {elapsed:.0f}s   vocab={len(counter):,}')
            gc.collect()

        title = article.get('title', '') or ''
        text  = (article.get('text', '') or '')[:char_cap]
        content = f'{title} {text}'
        for m in entity_pattern.findall(content):
            if len(m) >= 4 and not m.isupper():
                counter[m.lower()] += 1

    entities = {e for e, c in counter.items() if c >= 3}
    print(f'  ✓ {len(entities):,} entities (freq>=3) in {time()-t0:.0f}s')

    del counter; gc.collect()

    with open(ENTITY_CACHE, 'wb') as f:
        pickle.dump(entities, f)
    print(f'  ✓ Saved to {ENTITY_CACHE} ({ENTITY_CACHE.stat().st_size/1e6:.1f} MB)')
    return entities

if ENTITY_CACHE.exists():
    with open(ENTITY_CACHE, 'rb') as f:
        indian_entities = pickle.load(f)
    print(f'✓ Loaded {len(indian_entities):,} Indian entities from cache  '
          f'({ENTITY_CACHE.stat().st_size/1e6:.1f} MB)')
else:
    indian_entities = build_indian_entity_lexicon()


✓ Loaded 146,493 Indian entities from cache  (2.2 MB)


## 7 · Cache 4 — Metaphor detection model

We use `lwachowiak/Metaphor-Detection-XLMR`, a token-classification model
fine-tuned on the VUA Metaphor Corpus (Steen et al. 2010, ~200k tokens, MIPVU
gold standard). Model loading is **deferred** to Pass 2: at this point we only
define the factory.

In [ ]:
METAPHOR_BATCH_SIZE = 16  # Conservative for CPU; raise on GPU.

def load_metaphor_pipe():
    '''Factory — invoked only just before Pass 2 to keep Pass 1 lean.'''
    from transformers import pipeline as hf_pipeline
    return hf_pipeline(
        'token-classification',
        model='lwachowiak/Metaphor-Detection-XLMR',
        device=-1,
        aggregation_strategy='simple',  # subword → word-level
        ignore_labels=[],
    )

met_pipe = None
print('Metaphor pipe NOT yet loaded — will load just before Pass 2.')


Metaphor pipe NOT yet loaded — will load just before Pass 2.


## 8 · Static lexicons (compiled once)

In [ ]:
# ── Content-word POS set (Halliday 1985 lexical density) ─────────────────────
CONTENT_POS = {'NOUN', 'VERB', 'ADJ', 'ADV'}

# ── Simile patterns (D9.2) — Niculae & Danescu-Niculescu-Mizil 2014 ──────────
simile_matcher = Matcher(nlp.vocab)
simile_matcher.add('SIMILE', [
    [{'LOWER': 'like'}, {'POS': 'DET', 'OP': '?'}, {'POS': {'IN': ['NOUN','PROPN','ADJ']}}],
    [{'LOWER': 'as'}, {'POS': 'ADJ'}, {'LOWER': 'as'}],
    [{'LOWER': {'IN': ['resembles','resembling','similar']}}],
])

# ── Temporal markers (D9.4) ──────────────────────────────────────────────────
FORWARD_MARKERS  = {'then','after','next','later','finally','subsequently',
                    'eventually','afterwards','henceforth'}
BACKWARD_MARKERS = {'before','previously','earlier','prior','ago',
                    'used to','had been','in the past'}
TEMPORAL_ADVERBS = {
    'now','then','today','yesterday','tomorrow','soon','already','still',
    'yet','once','always','never','recently','currently','immediately',
    'eventually','finally','dawn','morning','afternoon','evening','night',
    'suddenly','meanwhile','afterwards','beforehand','henceforth','previously'
}

# ── Contractions (D9.5) ──────────────────────────────────────────────────────
CONTRACTION_RE = re.compile(
    r"\b(n't|'re|'ve|'ll|'d|'m|won't|can't|don't|didn't|couldn't|shouldn't"
    r"|wouldn't|isn't|aren't|wasn't|weren't|hasn't|haven't|it's|i'm|i've"
    r"|we're|they're|you're|he's|she's|that's|there's|let's|what's|who's)\b",
    re.IGNORECASE
)

# ── Pronouns (D9.5) — Biber 1988 ─────────────────────────────────────────────
FIRST  = {'i','me','my','mine','myself','we','us','our','ours','ourselves'}
SECOND = {'you','your','yours','yourself','yourselves'}
THIRD  = {'he','him','his','himself','she','her','hers','herself',
          'they','them','their','theirs','themselves','it','its','itself'}

# ── Implicature / metadiscourse markers (D9.6) ───────────────────────────────
#   Hyland 2005 (transitions, frame markers, hedges, boosters) +
#   Mann & Thompson 1988 (RST relation cues) +
#   Levinson 2000 (Q/I/M scalar/clausal triggers)
IMPLICATURE_MARKERS = {
    'but','however','although','though','yet','still','nevertheless','nonetheless',
    'despite','on the other hand','even so','admittedly','while','whereas','even though',
    'in spite of','because','since','therefore','thus','hence','consequently','so',
    'as a result','for this reason','accordingly','thereby','moreover','furthermore',
    'in addition','besides','additionally','also','perhaps','maybe','possibly','probably',
    'apparently','seemingly','it seems','it appears','supposedly','allegedly',
    'if','unless','provided that','as long as','in case','for example','for instance',
    'that is','namely','in other words',
}
IMPLICATURE_RE = re.compile(r'\b(' + '|'.join(re.escape(m) for m in IMPLICATURE_MARKERS) + r')\b')

# ── Cultural-entity helper sets (D9.7) ───────────────────────────────────────
NON_CULTURAL_WORDS = {
    'time','day','night','year','month','week','hour','minute',
    'today','yesterday','tomorrow','now','then','ago','later',
    'first','second','last','next','previous','current',
    'this','that','these','those','some','any','no','every',
    'all','both','each','either','neither','none',
    'very','too','also','just','only','even','such','same',
    'different','other','another','former','latter',
}
CULTURAL_LABELS = {'PERSON','GPE','LOC','ORG','NORP','FAC','EVENT','WORK_OF_ART','PROPN'}
TEMPORAL_LABELS = {'DATE','TIME'}

# ── Domain vocab ratio (D9.7) — Van Heuven et al. 2014 Zipf scale ────────────
ZIPF_THRESHOLD = 3.0  # words below this are "low-frequency / specialised"

print('All static lexicons compiled.')


All static lexicons compiled.


## 9 · Per-document feature extractor (24 features from spaCy; metaphor added in Pass 2)

In [ ]:
def extract_d9_per_doc(doc, raw_text):
    '''
    Extract 24 D9 features from a spaCy Doc + raw text.
    metaphor_density is added in Pass 2 (deferred for memory isolation).
    Returns a dict.
    '''
    out = {}

    sentences = [s.text for s in doc.sents]
    n_sents   = max(len(sentences), 1)
    tokens    = list(doc)
    n_tokens  = max(len(tokens), 1)
    alpha_tok = [t for t in tokens if t.is_alpha]
    n_alpha   = max(len(alpha_tok), 1)

    # ── 1. information_density_content_words (Halliday 1985) ─────────────────
    content_tokens = [t for t in tokens if t.pos_ in CONTENT_POS]
    out['information_density_content_words'] = len(content_tokens) / n_tokens

    # ── 2. propositional_density (CPIDR — Brown et al. 2008) ─────────────────
    try:
        from ideadensity import cpidr
        _, _, prop_dens, _ = cpidr(raw_text)
        out['propositional_density'] = float(prop_dens)
    except Exception:
        out['propositional_density'] = float('nan')

    # ── 3-4. mean_concreteness, mean_imageability ────────────────────────────
    # Brysbaert et al. 2014 (concreteness); Coltheart 1981 MRC (imageability).
    content_lemmas = [t.lemma_.lower() for t in tokens
                      if t.pos_ in CONTENT_POS and t.is_alpha and len(t.text) > 1]
    conc_scores = [merged_conc[w]  for w in content_lemmas if w in merged_conc]
    img_scores  = [mrc_img_dict[w] for w in content_lemmas if w in mrc_img_dict]

    out['mean_concreteness'] = float(np.mean(conc_scores)) if conc_scores else float('nan')
    out['mean_imageability'] = float(np.mean(img_scores))  if img_scores  else float('nan')

    # ── 5. simile_density (Niculae & Danescu-Niculescu-Mizil 2014) ───────────
    sim_matches = simile_matcher(doc)
    out['simile_density'] = len(sim_matches) / n_sents

    # ── 6. personification_density (Keh 2022 PINEAPPLE) ──────────────────────
    pers_count = 0
    for token in doc:
        if token.dep_ in ('nsubj', 'nsubjpass') and token.head.pos_ == 'VERB':
            if token.ent_type_ == 'PERSON':
                continue
            v_score = verb_scores.get(token.head.lemma_.lower(), 0.5)
            if v_score >= PERS_THRESHOLD:
                pers_count += 1
    out['personification_density'] = pers_count / n_sents

    # ── 7-9. Narrative structure ─────────────────────────────────────────────
    person_ents = {e.text.strip().title() for e in doc.ents if e.label_ == 'PERSON'}
    propn_heads = {ch.root.text.title() for ch in doc.noun_chunks if ch.root.pos_ == 'PROPN'}
    out['character_count'] = len(person_ents | propn_heads)

    seen, intro_per_sent = set(), []
    for sent in doc.sents:
        new = 0
        for ent in sent.ents:
            if ent.label_ == 'PERSON':
                name = ent.text.strip().title()
                if name not in seen:
                    seen.add(name); new += 1
        intro_per_sent.append(new)
    out['character_introduction_rate'] = float(np.mean(intro_per_sent)) if intro_per_sent else 0.0

    sent_sentiments = [vader.polarity_scores(s)['compound'] for s in sentences]
    out['story_arc_complexity'] = float(np.std(sent_sentiments)) if sent_sentiments else 0.0

    # ── 10. chronological_order_score ────────────────────────────────────────
    events = []
    for sent in doc.sents:
        sent_lower = sent.text.lower()
        has_back   = any(m in sent_lower for m in BACKWARD_MARKERS)
        root_verb  = next((t for t in sent if t.dep_ == 'ROOT' and t.pos_ == 'VERB'), None)
        if root_verb is None:
            continue
        tense = (root_verb.morph.get('Tense') or [''])[0].lower()
        if has_back:          tv = 0
        elif tense == 'past': tv = 1
        elif tense == 'pres': tv = 2
        elif tense == 'fut':  tv = 3
        else:                 tv = None
        if tv is not None:
            events.append(tv)

    if len(events) < 2:
        tl  = raw_text.lower()
        fwd = sum(1 for m in FORWARD_MARKERS  if m in tl)
        bwd = sum(1 for m in BACKWARD_MARKERS if m in tl)
        out['chronological_order_score'] = fwd / (fwd + bwd) if (fwd + bwd) else 0.5
    else:
        fwd = sum(1 for i in range(1, len(events)) if events[i] >= events[i-1])
        out['chronological_order_score'] = fwd / (len(events) - 1)

    # ── 11. temporal_expression_density ──────────────────────────────────────
    temporal_token_ids = set()
    for e in doc.ents:
        if e.label_ in ('DATE', 'TIME'):
            for t in e:
                temporal_token_ids.add(t.i)
    for t in doc:
        if t.lower_ in TEMPORAL_ADVERBS:
            temporal_token_ids.add(t.i)
    out['temporal_expression_density'] = len(temporal_token_ids) / n_sents

    # ── 12-13. tense_consistency, tense_switch_frequency ─────────────────────
    finite_verbs = [(t.text, (t.morph.get('Tense') or ['?'])[0])
                    for t in doc if t.pos_ == 'VERB' and t.morph.get('Tense')]
    if finite_verbs:
        tcounts = Counter(t for _, t in finite_verbs)
        dom = tcounts.most_common(1)[0][0]
        out['tense_consistency'] = tcounts[dom] / len(finite_verbs)
        sw = sum(1 for i in range(1, len(finite_verbs))
                 if finite_verbs[i][1] != finite_verbs[i-1][1])
        # Denominator is the number of ADJACENT PAIRS, not the number of verbs.
        out['tense_switch_frequency'] = sw / max(len(finite_verbs) - 1, 1)
    else:
        out['tense_consistency']      = 0.0
        out['tense_switch_frequency'] = 0.0

    # ── 14. formality_score (Heylighen & Dewaele 1999 F-score) ───────────────
    pos_counts = Counter(t.pos_ for t in doc if t.is_alpha)
    total_pos  = max(sum(pos_counts.values()), 1)
    pct = lambda tag: (pos_counts.get(tag, 0) / total_pos) * 100
    out['formality_score'] = (pct('NOUN') + pct('ADJ') + pct('ADP') + pct('DET')
                              - pct('PRON') - pct('VERB') - pct('ADV') - pct('INTJ')
                              + 100) / 2

    # ── 15. contraction_density ──────────────────────────────────────────────
    out['contraction_density'] = len(CONTRACTION_RE.findall(raw_text)) / n_alpha

    # ── 16-18. Pronoun densities (Biber 1988) ────────────────────────────────
    pron_tokens = [t for t in doc if t.pos_ == 'PRON']
    f_ct  = sum(1 for t in pron_tokens if t.lower_ in FIRST)
    s_ct  = sum(1 for t in pron_tokens if t.lower_ in SECOND)
    th_ct = sum(1 for t in pron_tokens if t.lower_ in THIRD)
    out['first_person_pronoun_density']  = f_ct  / n_alpha
    out['second_person_pronoun_density'] = s_ct  / n_alpha
    out['third_person_pronoun_density']  = th_ct / n_alpha

    # ── 19-21. Sentence-type densities ───────────────────────────────────────
    def _strip_q(s):
        s = s.strip()
        return s[1:-1] if s.startswith('"') and s.endswith('"') else s

    q   = sum(1 for s in sentences if _strip_q(s).endswith('?'))
    exc = sum(1 for s in sentences if _strip_q(s).endswith('!'))
    imp = 0
    for sent in doc.sents:
        for t in sent:
            if (t.dep_ == 'ROOT' and t.pos_ == 'VERB'
                    and not any(c.dep_ == 'nsubj' for c in t.children)):
                imp += 1
    out['question_density']    = q   / n_sents
    out['exclamation_density'] = exc / n_sents
    out['imperative_density']  = imp / n_sents

    # ── 22. implicature_marker_density (Hyland 2005) ─────────────────────────
    text_lower  = raw_text.lower()
    marker_hits = len(IMPLICATURE_RE.findall(text_lower))
    out['implicature_marker_density'] = marker_hits / n_sents

    # ── 23. cultural_entity_density (Bhasha-Wiki-Indic) ──────────────────────
    ner_entities = []
    for ent in doc.ents:
        tl = ent.text.lower()
        if tl in NON_CULTURAL_WORDS or len(ent.text) < 2 or ent.text.isdigit():
            continue
        if ent.label_ in TEMPORAL_LABELS:
            continue
        if ent.label_ in CULTURAL_LABELS:
            ner_entities.append(tl)
    propn_extra = []
    seen_lower  = set(ner_entities)
    for token in doc:
        if token.pos_ == 'PROPN':
            tl = token.text.lower()
            if tl in seen_lower or tl in NON_CULTURAL_WORDS or len(token.text) < 2:
                continue
            if token.text[0].isupper():
                propn_extra.append(tl)
                seen_lower.add(tl)
    all_entities = list(set(ner_entities + propn_extra))
    indian_found = [e for e in all_entities if e in indian_entities]
    out['cultural_entity_density'] = (len(indian_found) / len(all_entities)) if all_entities else 0.0

    # ── 24. domain_vocab_ratio (Van Heuven et al. 2014) ──────────────────────
    from wordfreq import zipf_frequency
    cl   = [t.lemma_.lower() for t in doc
            if t.pos_ in CONTENT_POS and t.is_alpha and len(t.text) > 2]
    rare = [w for w in cl if zipf_frequency(w, 'en') < ZIPF_THRESHOLD]
    out['domain_vocab_ratio'] = (len(rare) / len(cl)) if cl else 0.0

    return out

print('Per-doc extractor defined (24 features; metaphor_density added in Pass 2).')


Per-doc extractor defined (24 features; metaphor_density added in Pass 2).


## 10 · Pass 1 — spaCy + 24 features per document (memory-safe)

We truncate any essay to 50,000 chars before parsing. All D9 features are
densities or ratios (not totals), so a 50k-char prefix gives statistically
stable estimates on any narrative length. `batch_size=4` keeps only four spaCy
`Doc` objects alive at a time.

In [ ]:
MAX_CHARS = 50_000

texts_full = df_raw[TEXT_COL].astype(str).tolist()
indices    = df_raw[ID_COL].tolist()

n_trunc = sum(1 for t in texts_full if len(t) > MAX_CHARS)
if n_trunc:
    print(f'⚠ Truncating {n_trunc} essay(s) longer than {MAX_CHARS:,} chars.')
texts = [t[:MAX_CHARS] for t in texts_full]
del texts_full
gc.collect()

print(f'Pass 1: parsing {len(texts)} essays (batch_size=4 for low peak RAM)...')
t0      = time()
results = []
sentences_by_idx = {}

for i, doc in enumerate(nlp.pipe(texts, batch_size=4)):
    raw   = texts[i]
    idx   = indices[i]
    feats = extract_d9_per_doc(doc, raw)
    feats['index'] = idx
    results.append(feats)
    sentences_by_idx[idx] = [s.text for s in doc.sents]
    del doc
    if (i + 1) % 5 == 0:
        gc.collect()
    if (i + 1) % 10 == 0:
        elapsed = time() - t0
        rate    = (i + 1) / elapsed
        eta     = (len(texts) - i - 1) / rate
        print(f'  {i+1}/{len(texts)}   {elapsed:.1f}s   '
              f'{rate:.1f} docs/s   ETA {eta:.0f}s')

del texts; gc.collect()
print(f'✓ Pass 1 done in {time()-t0:.1f}s')


Pass 1: parsing 567 essays (batch_size=4 for low peak RAM)...
  10/567   11.4s   0.9 docs/s   ETA 638s
  20/567   20.2s   1.0 docs/s   ETA 553s
  30/567   27.5s   1.1 docs/s   ETA 492s
  40/567   32.7s   1.2 docs/s   ETA 431s
  50/567   39.5s   1.3 docs/s   ETA 408s
  60/567   46.5s   1.3 docs/s   ETA 393s
  70/567   53.8s   1.3 docs/s   ETA 382s
  80/567   56.3s   1.4 docs/s   ETA 343s
  90/567   59.7s   1.5 docs/s   ETA 317s
  100/567   63.1s   1.6 docs/s   ETA 295s
  110/567   66.2s   1.7 docs/s   ETA 275s
  120/567   68.8s   1.7 docs/s   ETA 256s
  130/567   71.8s   1.8 docs/s   ETA 241s
  140/567   75.4s   1.9 docs/s   ETA 230s
  150/567   78.4s   1.9 docs/s   ETA 218s
  160/567   81.1s   2.0 docs/s   ETA 206s
  170/567   84.1s   2.0 docs/s   ETA 196s
  180/567   87.8s   2.0 docs/s   ETA 189s
  190/567   91.0s   2.1 docs/s   ETA 181s
  200/567   93.9s   2.1 docs/s   ETA 172s
  210/567   97.4s   2.2 docs/s   ETA 166s
  220/567   101.7s   2.2 docs/s   ETA 160s
  230/567   104.9s   2

## 11 · Pass 2 — metaphor_density (token-level VUA-trained XLM-R)

Loads `lwachowiak/Metaphor-Detection-XLMR` (Wachowiak & Gromann 2023), runs
token classification over every sentence captured in Pass 1, and aggregates to
the document level as
`metaphor_density = #(tokens labelled LABEL_1) / #(scored tokens)` — i.e. the
proportion of words classified as metaphorical, following the VUA Metaphor
Corpus annotation scheme (Steen et al. 2010).

In [ ]:
# Free spaCy now that Pass 1 is complete — saves ~50 MB.
try:
    del nlp; gc.collect(); print('Freed spaCy pipeline.')
except NameError:
    pass

print('Loading metaphor pipe (deferred from §7)...')
try:
    met_pipe = load_metaphor_pipe()
    print('✓ Metaphor pipe ready')
except Exception as e:
    met_pipe = None
    print(f'⚠ Metaphor model failed to load: {e}')

metaphor_density_by_idx = {}

if met_pipe is None:
    for idx in indices:
        metaphor_density_by_idx[idx] = float('nan')
    print('Skipping metaphor pass (model unavailable).')
else:
    t1 = time()
    flat = []
    for idx in indices:
        for s in sentences_by_idx.get(idx, []):
            if s.strip():
                flat.append((idx, s))
    print(f'Pass 2 (token-level): {len(flat)} sentences across {len(indices)} essays')

    CHUNK_SIZE = 200
    agg = defaultdict(lambda: [0, 0])  # idx → [metaphor_count, total_count]

    for chunk_start in range(0, len(flat), CHUNK_SIZE):
        chunk = flat[chunk_start:chunk_start + CHUNK_SIZE]
        preds = met_pipe([s for _, s in chunk], batch_size=METAPHOR_BATCH_SIZE)

        for (idx, _), per_sent in zip(chunk, preds):
            for tok in per_sent:
                agg[idx][1] += 1
                lab = tok.get('entity_group', tok.get('entity', '')).strip()
                if lab == 'LABEL_1':
                    agg[idx][0] += 1

        del preds; gc.collect()
        elapsed = time() - t1
        done = min(chunk_start + CHUNK_SIZE, len(flat))
        eta = (len(flat) - done) * elapsed / max(done, 1)
        print(f'  {done}/{len(flat)} sentences   {elapsed:.0f}s elapsed   ETA {eta:.0f}s')

    for idx in indices:
        m, n = agg[idx]
        metaphor_density_by_idx[idx] = (m / n) if n else 0.0
    print(f'✓ Pass 2 done in {time()-t1:.1f}s')

# Free the metaphor model + sentence map.
try:
    del met_pipe
except NameError:
    pass
del sentences_by_idx
gc.collect()

# Merge metaphor_density into the per-doc records.
for r in results:
    r['metaphor_density'] = metaphor_density_by_idx.get(r['index'], float('nan'))

out_df = pd.DataFrame(results)
print(f'\\n✓ All 25 features assembled  ({out_df.shape})')


Freed spaCy pipeline.
Loading metaphor pipe (deferred from §7)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: lwachowiak/Metaphor-Detection-XLMR
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Metaphor pipe ready
Pass 2 (token-level): 20028 sentences across 567 essays
  200/20028 sentences   54s elapsed   ETA 5316s
  400/20028 sentences   85s elapsed   ETA 4153s
  600/20028 sentences   114s elapsed   ETA 3697s
  800/20028 sentences   145s elapsed   ETA 3487s
  1000/20028 sentences   182s elapsed   ETA 3467s
  1200/20028 sentences   211s elapsed   ETA 3318s
  1400/20028 sentences   241s elapsed   ETA 3212s
  1600/20028 sentences   274s elapsed   ETA 3154s
  1800/20028 sentences   304s elapsed   ETA 3074s
  2000/20028 sentences   333s elapsed   ETA 2998s
  2200/20028 sentences   363s elapsed   ETA 2942s
  2400/20028 sentences   393s elapsed   ETA 2888s
  2600/20028 sentences   420s elapsed   ETA 2819s
  2800/20028 sentences   451s elapsed   ETA 2773s
  3000/20028 sentences   484s elapsed   ETA 2749s
  3200/20028 sentences   512s elapsed   ETA 2692s
  3400/20028 sentences   540s elapsed   ETA 2642s
  3600/20028 sentences   570s elapsed   ETA 2603s
  3800/20028 sentences   603

## 12 · Diagnostics — sanity-check the extraction

In [ ]:
D9_NUMERIC = [
    # D9.1 — Lexical-Semantic Density
    'information_density_content_words',
    'propositional_density',
    'mean_concreteness',
    'mean_imageability',
    # D9.2 — Figurative Language
    'simile_density',
    'metaphor_density',
    'personification_density',
    # D9.3 — Narrative Structure
    'character_count',
    'character_introduction_rate',
    'story_arc_complexity',
    # D9.4 — Temporal & Tense
    'chronological_order_score',
    'temporal_expression_density',
    'tense_consistency',
    'tense_switch_frequency',
    # D9.5 — Register & Formality
    'formality_score',
    'contraction_density',
    'first_person_pronoun_density',
    'second_person_pronoun_density',
    'third_person_pronoun_density',
    # D9.6 — Pragmatic Inference
    'question_density',
    'exclamation_density',
    'imperative_density',
    'implicature_marker_density',
    # D9.7 — Cultural & Domain
    'cultural_entity_density',
    'domain_vocab_ratio',
]
assert len(D9_NUMERIC) == 25, f'Expected 25 features, got {len(D9_NUMERIC)}'
print(f'Feature count: {len(D9_NUMERIC)}')

print('\\nSummary statistics:')
print(out_df[D9_NUMERIC].describe().T[['mean','std','min','max']].round(3))

print('\\nMissing values per feature:')
miss = out_df[D9_NUMERIC].isna().sum()
print(miss[miss > 0] if miss.any() else '  (none)')

print('\\nZero-variance features (suspicious):')
zv = [c for c in D9_NUMERIC if out_df[c].nunique() <= 1]
print(zv if zv else '  (none)')

# Defensibility check: confirm no remaining perfectly-correlated pairs.
print('\\nPairwise correlation audit (|r| ≥ 0.95 between numeric features):')
corr = out_df[D9_NUMERIC].corr().abs()
high_pairs = []
for i in range(len(D9_NUMERIC)):
    for j in range(i+1, len(D9_NUMERIC)):
        r = corr.iloc[i, j]
        if r >= 0.95:
            high_pairs.append((D9_NUMERIC[i], D9_NUMERIC[j], r))
if high_pairs:
    for a, b, r in high_pairs:
        print(f'  ⚠ {a}  ↔  {b}   r = {r:.3f}')
else:
    print('  ✓ none — no redundant features')


Feature count: 25
\nSummary statistics:
                                      mean     std      min      max
information_density_content_words    0.410   0.037    0.300    0.510
propositional_density                0.492   0.028    0.409    0.605
mean_concreteness                    2.002   0.183    1.504    2.588
mean_imageability                  288.056  28.021  198.660  372.639
simile_density                       0.027   0.031    0.000    0.174
metaphor_density                     0.397   0.036    0.224    0.461
personification_density              0.136   0.082    0.000    0.469
character_count                     22.607  11.545    3.000   92.000
character_introduction_rate          0.186   0.140    0.000    0.960
story_arc_complexity                 0.367   0.070    0.202    0.570
chronological_order_score            0.784   0.076    0.545    1.000
temporal_expression_density          0.705   0.481    0.000    3.417
tense_consistency                    0.643   0.098    0.500    